# Iris classification – reference notebook

A complete, reproducible reference implementation for the Mage ML assignment. Uses the built-in Iris dataset.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## 1. Load data

In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame
feature_columns = list(iris.feature_names)
target_column = 'target'
df.head()

## 2. Prepare data (120 train / 30 test)

In [ ]:
X = df[feature_columns]
y = df[target_column]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
assert len(X_train) == 120
assert len(X_test) == 30
print('Train rows:', len(X_train), 'Test rows:', len(X_test))

## 3. Train and compare max_depth 1–5

In [ ]:
results = []
models = {}
for depth in range(1, 6):
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    models[depth] = (model, pred)
    results.append({'max_depth': depth, 'test_accuracy': acc, 'test_passed': acc >= 0.80})
results_df = pd.DataFrame(results)
results_df

## 4. Save predictions for selected model

In [ ]:
selected_depth = 3
model, predictions = models[selected_depth]
predictions_df = pd.DataFrame({'actual': y_test.to_numpy(), 'predicted': predictions})
predictions_df.to_csv('iris_predictions.csv', index=False)
predictions_df.head()

## 5. Evaluation branch outputs

In [ ]:
cm = confusion_matrix(y_test, predictions)
print('Confusion matrix:\n', cm)
print(classification_report(y_test, predictions, target_names=iris.target_names))
metrics = pd.DataFrame(classification_report(y_test, predictions, target_names=iris.target_names, output_dict=True)).transpose()
metrics.to_csv('iris_metrics.csv')
metrics

## 6. Data quality block (additional block)

In [ ]:
assert not df.isnull().any().any()
assert df['target'].nunique() == 3
print(f'Rows: {len(df)}; classes: {df.target.nunique()}; no missing values')